# 02 — Noise Synthesis
Generate OCR, ASR, and social media noisy versions of CoNLL-2003.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

import random
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict
from noise import apply_ocr_noise, apply_asr_noise, apply_social_noise

In [ ]:
dataset = load_dataset("conll2003", trust_remote_code=True)
label_names = dataset["train"].features["ner_tags"].feature.names
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}
print(f"Labels: {label_names}")
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])} | Test: {len(dataset['test'])}")

In [ ]:
NOISE_SEED = 42

def apply_noise_to_split(split, noise_fn, p=0.3):
    """Apply noise_fn to every example in a dataset split. Returns list of dicts."""
    noisy_examples = []
    for ex in split:
        tokens = list(ex["tokens"])
        tags = list(ex["ner_tags"])
        noisy_tokens, noisy_tags = noise_fn(tokens, tags, id2label, label2id, p=p, seed=None)
        noisy_examples.append({
            "tokens": noisy_tokens,
            "ner_tags": noisy_tags,
        })
    return noisy_examples

def build_noisy_dataset(dataset, noise_fn, p=0.3):
    """Build a DatasetDict with train/validation/test splits."""
    random.seed(NOISE_SEED)
    splits = {}
    for split_name in ["train", "validation", "test"]:
        examples = apply_noise_to_split(dataset[split_name], noise_fn, p=p)
        splits[split_name] = Dataset.from_list(examples)
    return DatasetDict(splits)

In [ ]:
print("Generating OCR noisy dataset...")
ocr_dataset = build_noisy_dataset(dataset, apply_ocr_noise)
print("Generating ASR noisy dataset...")
asr_dataset = build_noisy_dataset(dataset, apply_asr_noise)
print("Generating Social media noisy dataset...")
social_dataset = build_noisy_dataset(dataset, apply_social_noise)
print("Done.")

In [ ]:
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "noisy")

ocr_dataset.save_to_disk(os.path.join(DATA_DIR, "ocr"))
asr_dataset.save_to_disk(os.path.join(DATA_DIR, "asr"))
social_dataset.save_to_disk(os.path.join(DATA_DIR, "social"))
print(f"Saved to {DATA_DIR}/{{ocr,asr,social}}/")

## Examples: Before vs After

In [ ]:
def show_examples(orig_split, noisy_split, n=3, title=""):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    for i in range(n):
        orig = orig_split[i]
        noisy = noisy_split[i]
        orig_pairs = list(zip(orig["tokens"], [label_names[t] for t in orig["ner_tags"]]))
        noisy_pairs = list(zip(noisy["tokens"], [label_names[t] for t in noisy["ner_tags"]]))
        print(f"\nExample {i+1}:")
        print(f"  ORIG : {' '.join(f'{t}/{g}' for t,g in orig_pairs[:12])}")
        print(f"  NOISY: {' '.join(f'{t}/{g}' for t,g in noisy_pairs[:12])}")

show_examples(dataset["test"], ocr_dataset["test"], title="OCR Noise")
show_examples(dataset["test"], asr_dataset["test"], title="ASR Noise")
show_examples(dataset["test"], social_dataset["test"], title="Social Media Noise")

## Statistics

In [ ]:
def compute_stats(orig_split, noisy_split, name):
    orig_lens = [len(ex["tokens"]) for ex in orig_split]
    noisy_lens = [len(ex["tokens"]) for ex in noisy_split]

    changed = 0
    total_tokens = 0
    for orig, noisy in zip(orig_split, noisy_split):
        orig_toks = orig["tokens"]
        noisy_toks = noisy["tokens"]
        total_tokens += len(orig_toks)  # count ALL original tokens
        for j, ot in enumerate(orig_toks):
            # deleted tokens (noisy is shorter) count as changed
            if j >= len(noisy_toks) or ot.lower() != noisy_toks[j].lower():
                changed += 1

    return {
        "noise_type": name,
        "avg_orig_len": round(sum(orig_lens)/len(orig_lens), 2),
        "avg_noisy_len": round(sum(noisy_lens)/len(noisy_lens), 2),
        "pct_tokens_changed": round(100 * changed / max(total_tokens, 1), 2),
    }

stats = [
    compute_stats(dataset["test"], ocr_dataset["test"], "OCR"),
    compute_stats(dataset["test"], asr_dataset["test"], "ASR"),
    compute_stats(dataset["test"], social_dataset["test"], "Social"),
]
pd.DataFrame(stats)